# Production Line Capacity Analysis

**Goal:** Assign every produced item to a production line (Inline, PD Line, JB Line, Fogg, Blister) based on item code and bottle type, then analyze capacity by line.

**How this notebook works:**
- Each gray box below is a "cell" you can run
- Click a cell, then press **Shift+Enter** to run it and move to the next
- Or press **Ctrl+Enter** to run it and stay on it
- Run cells top to bottom — later cells depend on earlier ones
- Green text starting with `#` are comments (notes to yourself, not code)

## 1. Setup & Load Data

In [1]:
import polars as pl
from datetime import date
from pathlib import Path
import subprocess
import sys

def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'local_machine_scripts' / 'sage100sqlite').exists():
            return p
    raise RuntimeError('Could not find repo root (expected local_machine_scripts/sage100sqlite).')

ROOT = _find_repo_root(Path.cwd())
SAGE_DIR = ROOT / 'local_machine_scripts' / 'sage100sqlite'
DB_DIR = SAGE_DIR / 'db'

# Prefer @sage_data.db if present, else sage_data.db
DB_PATH = DB_DIR / '@sage_data.db'
if not DB_PATH.exists():
    DB_PATH = DB_DIR / 'sage_data.db'

YEAR = date.today().year - 1  # default: last year (e.g. 2025)
CSV_PATH = DB_DIR / f'line_capacity_sales_{YEAR}.csv'
EXTRACT_PY = SAGE_DIR / 'line_capacity_sales_extract.py'

if not CSV_PATH.exists():
    print(f'Missing base dataset: {CSV_PATH}')
    print('Generating it now from SQLite...')
    subprocess.run(
        [sys.executable, str(EXTRACT_PY), '--db', str(DB_PATH), '--year', str(YEAR)],
        check=True,
    )

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f'Base dataset still missing after generation attempt: {CSV_PATH}\n'
        f'Try running: python "{EXTRACT_PY}" --db "{DB_PATH}" --year {YEAR}'
    )

so_txns = pl.read_csv(CSV_PATH)

# Keep the extractor's line assignment for comparison.
if 'line' in so_txns.columns:
    so_txns = so_txns.rename({'line': 'line_script'})

# Notebook later does so_txns.drop('line'); add a placeholder if needed.
if 'line' not in so_txns.columns:
    so_txns = so_txns.with_columns(pl.lit(None).alias('line'))

# Rule 1 item codes (Fogg)
fogg_path = SAGE_DIR / 'fogg_item_codes.csv'
fogg_codes = [c.strip() for c in fogg_path.read_text(encoding='utf-8').splitlines() if c.strip()]
existing_rules = pl.DataFrame({'itemcode': fogg_codes, 'line': ['Fogg'] * len(fogg_codes)})

print(f'DB:           {DB_PATH}')
print(f'Dataset CSV:  {CSV_PATH}')
print(f'SO rows:      {so_txns.shape[0]:,} ({so_txns["itemcode"].n_unique()} unique items)')
print(f'Rule 1 codes: {existing_rules.shape[0]} item codes (Fogg)')

# Bottle receipts (supply-side) dataset
BOTTLE_ITEM_CODE = "022001"  # BOTTLE-RIGS NATURAL  120gm
BR_CSV_PATH = DB_DIR / f"bottle_receipts_{BOTTLE_ITEM_CODE}_{YEAR}.csv"
BR_EXTRACT_PY = SAGE_DIR / "line_capacity_bottle_receipts_extract.py"

if not BR_CSV_PATH.exists():
    print(f"Missing bottle receipts dataset: {BR_CSV_PATH}")
    print("Generating it now from SQLite...")
    subprocess.run(
        [
            sys.executable,
            str(BR_EXTRACT_PY),
            "--db",
            str(DB_PATH),
            "--year",
            str(YEAR),
            "--item",
            str(BOTTLE_ITEM_CODE),
        ],
        check=True,
    )

if not BR_CSV_PATH.exists():
    raise FileNotFoundError(
        f'Bottle receipts dataset still missing after generation attempt: {BR_CSV_PATH}\n'
        f'Try running: python "{BR_EXTRACT_PY}" --db "{DB_PATH}" --year {YEAR} --item {BOTTLE_ITEM_CODE}'
    )

br_txns = pl.read_csv(BR_CSV_PATH)
print(f"BR rows:      {br_txns.shape[0]:,} ({br_txns['transactiondate_iso'].n_unique()} unique days)")


DB:           C:\Users\jdavis\Documents\kpk-app\local_machine_scripts\sage100sqlite\db\sage_data.db
Dataset CSV:  C:\Users\jdavis\Documents\kpk-app\local_machine_scripts\sage100sqlite\db\line_capacity_sales_2025.csv
SO rows:      4,531 (1020 unique items)
Rule 1 codes: 112 item codes (Fogg)
BR rows:      194 (172 unique days)


## 2. Line Assignment Rules

Rules are applied in priority order:
1. **Item code lookup** — explicit item-to-line mapping (currently Fogg aerosol items)
2. **Product description overrides** — keywords in `itemcodedesc` force a line regardless of bottle:
   - "water shock" or "water treatment" → **PD Line**
   - "teak oil" in gallon/64oz+ → **Inline** (overrides the normal JB Line gallon rule)
3. **Bottle type lookup** — bottle name maps to a line
4. **Fallback** — anything left gets `"UNASSIGNED"`

In [2]:
# ============================================================
# RULE 2: Pattern + size overrides
# ============================================================
# Applied AFTER item code lookup, BEFORE bottle-type map.
# Later entries in this list take priority over earlier ones,
# so put general rules first and specific exceptions after.
#
# Each entry: (column, pattern, line, size_expr)
#   - column: "itemcodedesc", "Bottle", or None (match everything)
#   - pattern: substring to match (lowercased), ignored if column is None
#   - line: which production line
#   - size_expr: None = all sizes, or a native Polars expression

DESCRIPTION_OVERRIDES = [
    # --- General size rules (lowest priority, listed first) ---
    (None,           None,              "JB Line", pl.col("bottle_oz") == 64),   # all 64oz -> JB

    # --- Product-specific exceptions (override the general rules above) ---
    ("itemcodedesc", "water shock",     "PD Line", None),                        # any size
    ("itemcodedesc", "water treatment", "PD Line", None),                        # any size
    ("itemcodedesc", "water freshener", "PD Line", None),                        # any size
    ("itemcodedesc", "salt off",        "PD Line", None),                        # any size
    ("itemcodedesc", "salt remov",      "PD Line", None),                        # any size (Seachoice)
    ("itemcodedesc", "salt rmv",        "PD Line", None),                        # any size (Australia)
    ("itemcodedesc", "salt off",        "JB Line", pl.col("bottle_oz") >= 64),   # 64oz+ Salt Off runs on JB
    ("itemcodedesc", "salt remov",      "JB Line", pl.col("bottle_oz") >= 64),   # 64oz+ Salt Remover runs on JB
    ("itemcodedesc", "salt rmv",        "JB Line", pl.col("bottle_oz") >= 64),   # 64oz+ Salt Rmv runs on JB
    ("itemcodedesc", "teak oil",        "Inline",  pl.col("bottle_oz") >= 64),   # 64oz+ only
    ("Bottle",       "startron",        "Inline",  pl.col("bottle_oz") < 128),   # non-gallon
]

# ============================================================
# RULE 1: Item code -> Line  (explicit overrides, highest priority)
# ============================================================
EXTRA_ITEM_RULES = {
    # "SOME-ITEMCODE": "JB Line",
}

item_to_line = dict(zip(
    existing_rules["itemcode"].to_list(),
    existing_rules["line"].to_list()
))
item_to_line.update(EXTRA_ITEM_RULES)

# Build the override as a single Polars expression chain.
# Later entries wrap earlier ones, so they take priority.
desc_override_expr = pl.lit(None).cast(pl.Utf8)
for _col, _pat, _line, _size in DESCRIPTION_OVERRIDES:
    if _col is not None:
        _cond = pl.col(_col).str.to_lowercase().str.contains(_pat)
    else:
        _cond = pl.lit(True)
    if _size is not None:
        _cond = _cond & _size
    desc_override_expr = pl.when(_cond).then(pl.lit(_line)).otherwise(desc_override_expr)
desc_override_expr = desc_override_expr.alias("line_from_desc")

print(f"Item-level rules: {len(item_to_line)} total")
print(f"Override patterns: {len(DESCRIPTION_OVERRIDES)} (later entries win)")

# Preview matches
for _col, _pat, _line, _size in DESCRIPTION_OVERRIDES:
    q = so_txns.lazy()
    if _col is not None:
        q = q.filter(pl.col(_col).str.to_lowercase().str.contains(_pat))
    if _size is not None:
        q = q.filter(_size)
    result = q.select(pl.col("itemcode").n_unique().alias("items"), pl.len().alias("rows")).collect()
    col_label = f'{_col} ~ "{_pat}"' if _col else "all items"
    size_label = f" + size filter" if _size is not None else ""
    print(f"  {col_label}{size_label} -> {_line}: {result['items'][0]} items, {result['rows'][0]} rows")


Item-level rules: 112 total
Override patterns: 12 (later entries win)
  all items + size filter -> JB Line: 26 items, 142 rows
  itemcodedesc ~ "water shock" -> PD Line: 3 items, 6 rows
  itemcodedesc ~ "water treatment" -> PD Line: 0 items, 0 rows
  itemcodedesc ~ "water freshener" -> PD Line: 4 items, 16 rows
  itemcodedesc ~ "salt off" -> PD Line: 23 items, 147 rows
  itemcodedesc ~ "salt remov" -> PD Line: 3 items, 18 rows
  itemcodedesc ~ "salt rmv" -> PD Line: 4 items, 9 rows
  itemcodedesc ~ "salt off" + size filter -> JB Line: 5 items, 42 rows
  itemcodedesc ~ "salt remov" + size filter -> JB Line: 2 items, 15 rows
  itemcodedesc ~ "salt rmv" + size filter -> JB Line: 1 items, 2 rows
  itemcodedesc ~ "teak oil" + size filter -> Inline: 6 items, 25 rows
  Bottle ~ "startron" + size filter -> Inline: 58 items, 243 rows


In [3]:
# ============================================================
# RULE 3: Bottle type -> Line
# ============================================================
# Fallback after item codes and pattern/size overrides.
# 64oz bottles are handled by the size override (Rule 2) so
# they don't appear here.
#
# PD Line = spray triggers ONLY. All other cap types
# (flip top, pump top, screw cap, tip-and-pour, vent) → Inline.

BOTTLE_TO_LINE = {
    # --- Blister (1oz) ---
    "BOTTLE-1oz Startron teal blue":       "Blister",

    # --- Inline (small bottles + all non-sprayer 16oz/32oz) ---
    "BOTTLE-4oz Round transblue pvc":       "Inline",
    "BOTTLE-4oz round trans orange":        "Inline",
    "BOTTLE-8oz STARTRON TRNSBLUE":         "Inline",
    "BOTTLE-8oz ST Trans-Org PVC":          "Inline",
    "BOTTLE-8oz ROUND PVC Clear":           "Inline",
    "BOTTLE-8oz ROUND natural HDPE":        "Inline",
    "BOTTLE-8oz PVC OVAL WHITE":            "Inline",
    "BOTTLE-8oz ST TRANS BLACK PVC":        "Inline",
    "BOTTLE-8oz STARTRON WHITE PVC":        "Inline",
    "BOTTLE-8oz STARTRON CLEAR":            "Inline",
    "BOTTLE-8oz KPK Amsoil clear pv":       "Inline",
    "BOTTLE-16oz ST Trans-Org PVC":         "Inline",
    "BOTTLE-16oz STARTRN TRNSBLUE P":       "Inline",
    "BOTTLE-16oz Stron Trans Black":        "Inline",
    "BOTTLE-16oz PVC, WHITE":               "Inline",
    "BOTTLE-16oz PVC, CLEAR":               "Inline",
    "BOTTLE-16oz BLACK OVAL PVC":           "Inline",
    "BOTTLE-16oz PVC, SIKA BLUE":           "Inline",
    "BOTTLE-16oz TRANSBLUE OVAL PVC":       "Inline",
    "BOTTLE-32oz STARTR TRNS BLUE P":       "Inline",
    "BOTTLE-32oz PVC, CLEAR":               "Inline",
    "BOTTLE-32oz PVC, WHITE OVAL":          "Inline",
    "BOTTLE-32oz White PET Oval":           "Inline",
    "BOTTLE-32oz Natural hd round":         "Inline",
    "BOTTLE-32oz NAT handled round":        "Inline",
    "BOTTLE-32oz Nat HD Tip/Pour":          "Inline",
    "BOTTLE-32oz PVC, SIKA BLUE":           "Inline",
    "BOTTLE-32oz TRANSBLUE OVAL PVC":       "Inline",

    # --- PD Line (sprayers, triggers, oil quarts) ---
    "BOTTLE-2oz round white PET":           "PD Line",
    "BOTTLE-16oz ROUND PVC Clear":          "PD Line",
    "BOTTLE-16oz NAT ROUND HDPE":           "PD Line",
    "BOTTLE-16oz SIKA BLUE PVC SPRY":       "PD Line",
    "BOTTLE-16oz WHITE PVC SPRAYER":        "PD Line",
    "BOTTLE-16oz BLACK PVC SPRYR":          "PD Line",
    "BOTTLE-22oz White pvc":                "PD Line",
    "BOTTLE-22oz White PET NO S spr":       "PD Line",
    "BOTTLE-22oz Sika Blue":                "PD Line",
    "BOTTLE-22oz Black":                    "PD Line",
    "BOTTLE-22oz Cyan C":                   "PD Line",
    "BOTTLE-22oz Clear pvc":                "PD Line",
    "BOTTLE-22oz Green375C":                "PD Line",
    "BOTTLE-22oz RED 185C":                 "PD Line",
    "BOTTLE-32oz spray White":              "PD Line",
    "BOTTLE-32oz sprayer Sika":             "PD Line",
    "BOTTLE-32oz sprayer Clear":            "PD Line",
    'BOTTLE-32oz White NO "S" spray':       "PD Line",
    "BOTTLE-32oz Fstyle CLEAR PVC":         "PD Line",
    "BOTTLE-32oz Fstyle WHITE PVC 3":       "PD Line",
    "BOTTLE-32oz OIL QUART-BLUE":           "PD Line",
    "BOTTLE-32oz OIL QUART-BLACK":          "PD Line",
    "BOTTLE-32oz OIL QUART-NATURAL":        "PD Line",

    # --- JB Line (gallons, F-style jugs) ---
    "BOTTLE-F-STYLE WHITE/WHITE":           "JB Line",
    "BOTTLE-F-STYLE BLACK GALLON":          "JB Line",
    "BOTTLE-F-STYLE BLUE GALLON":           "JB Line",
    "BOTTLE-F-STYLE NATURAL":               "JB Line",
    "BOTTLE-RIG NATURAL 180 GR":            "JB Line",
    "BOTTLE-RIG White Gallon 140gra":       "JB Line",
    "BOTTLE-RIGS NATURAL  120gm":           "JB Line",
    "BOTTLE-GAL TRANSP BLUE PVC":           "JB Line",
    "BOTTLE-GAL SIKA BLUE PVC (38/4":       "JB Line",
    "BOTTLE-GAL CLEAR PVC":                 "JB Line",
    "BOTTLE-GAL WHITE PVC (38/400)":        "JB Line",
    "BOTTLE-GAL Orange 165C PVC (38":       "JB Line",
    "BOTTLE-2.5 GAL BLACK":                 "JB Line",
}

print(f"Bottle-level rules: {len(BOTTLE_TO_LINE)}")

Bottle-level rules: 65


## 3. Apply Rules

In [4]:
def assign_line(df: pl.DataFrame) -> pl.DataFrame:
    """Apply line assignment rules in priority order:
    1. Item code exact match
    2. Description overrides (keyword + optional size filter)
    3. Bottle type map
    4. UNASSIGNED fallback
    """
    item_map = pl.LazyFrame({
        "itemcode": list(item_to_line.keys()),
        "line_from_item": list(item_to_line.values()),
    })
    bottle_map = pl.LazyFrame({
        "Bottle": list(BOTTLE_TO_LINE.keys()),
        "line_from_bottle": list(BOTTLE_TO_LINE.values()),
    })

    return (
        df.lazy()
        .join(item_map, on="itemcode", how="left")
        .join(bottle_map, on="Bottle", how="left")
        .with_columns(desc_override_expr)
        .with_columns(
            pl.coalesce("line_from_item", "line_from_desc", "line_from_bottle")
              .fill_null("UNASSIGNED")
              .alias("line")
        )
        .drop("line_from_item", "line_from_desc", "line_from_bottle")
        .collect()
    )

so = assign_line(so_txns.drop("line"))

print("=== SO Transactions ===")
print(so.group_by("line").len().sort("len", descending=True))

# If the base dataset came from line_capacity_sales_extract.py, compare its line assignment
# (line_script) to what the notebook rules produce.
if "line_script" in so.columns:
    mismatches = so.filter(pl.col("line_script") != pl.col("line"))
    if mismatches.height:
        print()
        print("=== Notebook vs Extractor Line Differences ===")
        print(f"Mismatched rows: {mismatches.height:,}")
        print(
            mismatches
            .group_by(["line_script", "line"])
            .len()
            .sort("len", descending=True)
        )
    else:
        print()
        print("Notebook line assignments match extractor (no mismatches).")


# Quick sanity check: Salt Off 64oz+ should be JB Line
salt_off_large = (
    so.lazy()
    .filter(
        pl.col("itemcodedesc").str.to_lowercase().str.contains("salt off")
        & (pl.col("bottle_oz") >= 64)
    )
    .select(["itemcode", "itemcodedesc", "Bottle", "bottle_oz", "line_script", "line"])
    .unique()
    .collect()
)

if salt_off_large.height:
    print()
    print("=== Salt Off (64oz+/gallon) Line Check ===")
    print(salt_off_large.group_by(["line_script", "line"]).len().sort("len", descending=True))
    not_jb = salt_off_large.filter(pl.col("line") != "JB Line")
    if not_jb.height:
        print()
        print("Items NOT on JB Line (per notebook rules):")
        with pl.Config(tbl_rows=50, tbl_width_chars=200):
            print(not_jb.sort(["bottle_oz", "itemcode"]))
else:
    print()
    print("No Salt Off 64oz+ rows found in dataset.")


=== SO Transactions ===
shape: (5, 2)
┌────────────┬──────┐
│ line       ┆ len  │
│ ---        ┆ ---  │
│ str        ┆ u32  │
╞════════════╪══════╡
│ Inline     ┆ 1474 │
│ PD Line    ┆ 1407 │
│ JB Line    ┆ 1029 │
│ Fogg       ┆ 620  │
│ UNASSIGNED ┆ 1    │
└────────────┴──────┘

=== Notebook vs Extractor Line Differences ===
Mismatched rows: 117
shape: (3, 3)
┌─────────────┬─────────┬─────┐
│ line_script ┆ line    ┆ len │
│ ---         ┆ ---     ┆ --- │
│ str         ┆ str     ┆ u32 │
╞═════════════╪═════════╪═════╡
│ PD Line     ┆ JB Line ┆ 59  │
│ UNASSIGNED  ┆ PD Line ┆ 36  │
│ UNASSIGNED  ┆ Inline  ┆ 22  │
└─────────────┴─────────┴─────┘

=== Salt Off (64oz+/gallon) Line Check ===
shape: (1, 3)
┌─────────────┬─────────┬─────┐
│ line_script ┆ line    ┆ len │
│ ---         ┆ ---     ┆ --- │
│ str         ┆ str     ┆ u32 │
╞═════════════╪═════════╪═════╡
│ PD Line     ┆ JB Line ┆ 5   │
└─────────────┴─────────┴─────┘


## 4. Find the Gaps

These are items still marked `UNASSIGNED`. Use this to figure out what rules to add.

In [5]:
# Unassigned SO items grouped by bottle type
so_gaps = (
    so.lazy()
    .filter(pl.col("line") == "UNASSIGNED")
    .group_by("Bottle")
    .agg(
        pl.len().alias("rows"),
        pl.col("itemcode").n_unique().alias("unique_items"),
        pl.col("transactionqty").sum().alias("total_qty"),
    )
    .sort("rows", descending=True)
    .collect()
)

print("Unassigned SO rows by bottle type:")
with pl.Config(tbl_rows=50):
    print(so_gaps)

Unassigned SO rows by bottle type:
shape: (1, 4)
┌────────────────────────────────┬──────┬──────────────┬───────────┐
│ Bottle                         ┆ rows ┆ unique_items ┆ total_qty │
│ ---                            ┆ ---  ┆ ---          ┆ ---       │
│ str                            ┆ u32  ┆ u32          ┆ f64       │
╞════════════════════════════════╪══════╪══════════════╪═══════════╡
│ BOTTLE-32oz Nat HD Tip/Pour 28 ┆ 1    ┆ 1            ┆ -64.0     │
└────────────────────────────────┴──────┴──────────────┴───────────┘


In [6]:
# Drill into a specific bottle type to see the actual items
# Change the string below to explore different bottle types
BOTTLE_TO_INSPECT = "BOTTLE-16oz NAT ROUND HDPE"

detail = (
    so.lazy()
    .filter(
        (pl.col("line") == "UNASSIGNED") &
        (pl.col("Bottle") == BOTTLE_TO_INSPECT)
    )
    .group_by("itemcode", "itemcodedesc", "Bottle", "bottle_oz")
    .agg(pl.col("transactionqty").sum().alias("total_qty"))
    .sort("total_qty", descending=True)
    .collect()
)

with pl.Config(tbl_rows=100, tbl_width_chars=200):
    print(detail)

shape: (0, 5)
┌──────────┬──────────────┬────────┬───────────┬───────────┐
│ itemcode ┆ itemcodedesc ┆ Bottle ┆ bottle_oz ┆ total_qty │
│ ---      ┆ ---          ┆ ---    ┆ ---       ┆ ---       │
│ str      ┆ str          ┆ str    ┆ f64       ┆ f64       │
╞══════════╪══════════════╪════════╪═══════════╪═══════════╡
└──────────┴──────────────┴────────┴───────────┴───────────┘


### Blend & Cap Investigation

For bottle types where the line depends on the blend, this shows the blend and cap components from the BOM for each unassigned item.

In [7]:
import sqlite3

# Load BOM from SQLite (BM_BillDetail joined to CI_Item for component descriptions)
con = sqlite3.connect(str(DB_PATH))
cur = con.cursor()
cur.execute(
    """
    select
        bd.BillNo as item_code,
        bd.ComponentItemCode as component_item_code,
        ci.ItemCodeDesc as component_item_description
    from BM_BillDetail bd
    left join CI_Item ci
        on ci.ItemCode = bd.ComponentItemCode
    """
)
rows = cur.fetchall()
con.close()

bom = pl.DataFrame(
    rows,
    schema=['item_code', 'component_item_code', 'component_item_description'],
    orient="row",
)

print(f'BOM rows: {bom.shape[0]:,}')


BOM rows: 23,128


## 6. Line Capacity Analysis

Once your rules are solid, these cells give you production volume by line.

## 5. Preview Assigned Data

Spot-check the assignments before doing any analysis. Nothing is written to disk — this is just output in your browser.

In [8]:
# Preview: sample of assigned rows per line
for line_name in ["Inline", "PD Line", "JB Line", "Fogg", "Blister"]:
    sample = (
        so.filter(pl.col("line") == line_name)
        .select(["itemcode", "itemcodedesc", "Bottle", "bottle_oz", "line"])
        .unique()
        .head(5)
    )
    print(f"\n--- {line_name} (sample) ---")
    with pl.Config(tbl_width_chars=200):
        print(sample)

# Show assignment coverage
total = so.height
assigned = so.filter(pl.col("line") != "UNASSIGNED").height
pct = assigned / total * 100
print(f"\nCoverage: {assigned:,} / {total:,} rows assigned ({pct:.1f}%)")
print(f"Remaining: {total - assigned:,} rows need rules")


--- Inline (sample) ---
shape: (5, 5)
┌──────────┬────────────────────────────────┬────────────────────────────────┬───────────┬────────┐
│ itemcode ┆ itemcodedesc                   ┆ Bottle                         ┆ bottle_oz ┆ line   │
│ ---      ┆ ---                            ┆ ---                            ┆ ---       ┆ ---    │
│ str      ┆ str                            ┆ str                            ┆ f64       ┆ str    │
╞══════════╪════════════════════════════════╪════════════════════════════════╪═══════════╪════════╡
│ 11007283 ┆ WM Headchem Trt Mint 4pk-4-8oz ┆ BOTTLE-8oz ROUND PVC Clear     ┆ 8.0       ┆ Inline │
│ 18604P   ┆ Quick Fix 6-4oz                ┆ BOTTLE-4oz Round transblue pvc ┆ 4.0       ┆ Inline │
│ 086032   ┆ Prem Restorer Wax 6-32oz       ┆ BOTTLE-32oz PVC, WHITE OVAL    ┆ 32.0      ┆ Inline │
│ 15023773 ┆ WM Boat  Wash & Shine 12/32oz  ┆ BOTTLE-32oz White PET Oval     ┆ 32.0      ┆ Inline │
│ 920016   ┆ ST Gas PDQ Display 3-16oz      ┆ BOTTLE-16oz STA

In [9]:
# SO data summary by line
line_summary = (
    so.lazy()
    .filter(pl.col("line") != "UNASSIGNED")
    .group_by("line")
    .agg(
        pl.len().alias("total_txns"),
        pl.col("itemcode").n_unique().alias("unique_items"),
        pl.col("transactionqty").sum().alias("total_units"),
        pl.col("extendedcost").sum().round(2).alias("total_cost"),
    )
    .sort("total_units", descending=True)
    .collect()
)

print("Production by line (SO transactions, 2025):")
print(line_summary)

Production by line (SO transactions, 2025):
shape: (4, 5)
┌─────────┬────────────┬──────────────┬─────────────┬────────────┐
│ line    ┆ total_txns ┆ unique_items ┆ total_units ┆ total_cost │
│ ---     ┆ ---        ┆ ---          ┆ ---         ┆ ---        │
│ str     ┆ u32        ┆ u32          ┆ f64         ┆ f64        │
╞═════════╪════════════╪══════════════╪═════════════╪════════════╡
│ JB Line ┆ 1029       ┆ 223          ┆ -175092.0   ┆ -4.4521e6  │
│ PD Line ┆ 1407       ┆ 332          ┆ -315626.0   ┆ -3.8812e6  │
│ Inline  ┆ 1474       ┆ 414          ┆ -418004.96  ┆ -4.7234e6  │
│ Fogg    ┆ 620        ┆ 50           ┆ -690681.0   ┆ -6.2542e6  │
└─────────┴────────────┴──────────────┴─────────────┴────────────┘


In [10]:
# Monthly SO volume by line
monthly_pivot = (
    so.lazy()
    .filter(pl.col("line") != "UNASSIGNED")
    .with_columns(
        (pl.col("transactiondate") % 10000 // 100).cast(pl.Int32).alias("month")
    )
    .group_by("line", "month")
    .agg(
        pl.col("transactionqty").sum().alias("units"),
    )
    .collect()
    .pivot(on="line", index="month", values="units")
    .fill_null(0)
    .sort("month")
)

print("Monthly units by line:")
with pl.Config(tbl_width_chars=200):
    print(monthly_pivot)

Monthly units by line:
shape: (5, 5)
┌───────┬──────────┬───────────┬───────────┬───────────┐
│ month ┆ JB Line  ┆ Fogg      ┆ PD Line   ┆ Inline    │
│ ---   ┆ ---      ┆ ---       ┆ ---       ┆ ---       │
│ i32   ┆ f64      ┆ f64       ┆ f64       ┆ f64       │
╞═══════╪══════════╪═══════════╪═══════════╪═══════════╡
│ 56    ┆ -22651.0 ┆ -48482.0  ┆ -35226.0  ┆ -46368.96 │
│ 57    ┆ -70758.0 ┆ -109901.0 ┆ -125709.0 ┆ -145634.0 │
│ 58    ┆ -31014.0 ┆ -230168.0 ┆ -84220.0  ┆ -110227.0 │
│ 59    ┆ -40776.0 ┆ -273095.0 ┆ -51542.0  ┆ -89363.0  │
│ 60    ┆ -9893.0  ┆ -29035.0  ┆ -18929.0  ┆ -26412.0  │
└───────┴──────────┴───────────┴───────────┴───────────┘


In [11]:
# What bottle sizes run on each line?
size_mix = (
    so.lazy()
    .filter(pl.col("line") != "UNASSIGNED")
    .group_by("line", "bottle_oz")
    .agg(pl.col("transactionqty").sum().alias("units"))
    .sort("line", "bottle_oz")
    .collect()
)

with pl.Config(tbl_rows=60):
    print(size_mix)

shape: (15, 3)
┌─────────┬───────────┬────────────┐
│ line    ┆ bottle_oz ┆ units      │
│ ---     ┆ ---       ┆ ---        │
│ str     ┆ f64       ┆ f64        │
╞═════════╪═══════════╪════════════╡
│ Fogg    ┆ 128.0     ┆ -690681.0  │
│ Inline  ┆ 1.0       ┆ -3822.0    │
│ Inline  ┆ 4.0       ┆ -3240.0    │
│ Inline  ┆ 8.0       ┆ -132097.0  │
│ Inline  ┆ 16.0      ┆ -172348.0  │
│ Inline  ┆ 32.0      ┆ -104926.96 │
│ Inline  ┆ 128.0     ┆ -1571.0    │
│ JB Line ┆ 64.0      ┆ -31833.0   │
│ JB Line ┆ 128.0     ┆ -142642.0  │
│ JB Line ┆ 320.0     ┆ -617.0     │
│ PD Line ┆ 2.0       ┆ -498.0     │
│ PD Line ┆ 8.0       ┆ -1706.0    │
│ PD Line ┆ 16.0      ┆ -65448.0   │
│ PD Line ┆ 22.0      ┆ -111783.0  │
│ PD Line ┆ 32.0      ┆ -136191.0  │
└─────────┴───────────┴────────────┘


## 6b. Gallon Volume Analysis

Convert `qty_in_oz` to gallons (128 oz/gal) and look at overall, monthly, and weekly averages per line.

In [12]:
from datetime import date

so_gal = (
    so.lazy()
    .with_columns(
        ((pl.col("transactiondate").cast(pl.Int32) - 25569) * 86_400_000_000)
            .cast(pl.Datetime("us")).cast(pl.Date).alias("date"),
        (pl.col("qty_in_oz") / 128).alias("gallons"),
    )
    .with_columns(
        pl.col("date").dt.month().alias("month"),
        pl.col("date").dt.iso_year().alias("yr"),
        pl.col("date").dt.week().alias("week"),
    )
    .collect()
)

# Overall totals by line
line_totals = (
    so_gal.lazy()
    .group_by("line")
    .agg(
        pl.len().alias("records"),
        pl.col("gallons").sum().round(0).alias("total_gal"),
    )
    .sort("total_gal", descending=True)
    .with_columns(
        (pl.col("total_gal") / pl.col("total_gal").sum() * 100).round(1).alias("pct"),
    )
    .collect()
)

grand_total = line_totals["total_gal"].sum()
print(f"=== 2025 Total Volume: {grand_total:,.0f} gallons ===\n")
with pl.Config(tbl_width_chars=120, thousands_separator=","):
    print(line_totals)

=== 2025 Total Volume: 5,385,348 gallons ===

shape: (5, 4)
┌────────────┬─────────┬────────────┬──────┐
│ line       ┆ records ┆ total_gal  ┆ pct  │
│ ---        ┆ ---     ┆ ---        ┆ ---  │
│ str        ┆ u32     ┆ f64        ┆ f64  │
╞════════════╪═════════╪════════════╪══════╡
│ Fogg       ┆ 620     ┆ 4.144086e6 ┆ 77.0 │
│ JB Line    ┆ 1,029   ┆ 607,719.0  ┆ 11.3 │
│ PD Line    ┆ 1,407   ┆ 321,821.0  ┆ 6.0  │
│ Inline     ┆ 1,474   ┆ 311,530.0  ┆ 5.8  │
│ UNASSIGNED ┆ 1       ┆ 192.0      ┆ 0.0  │
└────────────┴─────────┴────────────┴──────┘


In [13]:
# Monthly gallons by line
monthly_gal = (
    so_gal.lazy()
    .group_by("line", "month")
    .agg(pl.col("gallons").sum().round(0).alias("gal"))
    .collect()
    .pivot(on="line", index="month", values="gal")
    .fill_null(0)
    .sort("month")
)

n_months = monthly_gal.height

print("=== Monthly Gallons by Line ===")
with pl.Config(tbl_width_chars=200, tbl_rows=n_months, thousands_separator=","):
    print(monthly_gal)

# Monthly averages
print(f"\n=== Monthly AVERAGE ({n_months} months) ===")
avg_row = {col: f"{monthly_gal[col].mean():,.0f}" for col in monthly_gal.columns if col != "month"}
for line, avg in sorted(avg_row.items(), key=lambda x: -float(x[1].replace(",",""))):
    print(f"  {line:12s}  {avg:>10s} gal/month")

=== Monthly Gallons by Line ===
shape: (12, 6)
┌───────┬──────────┬──────────┬───────────┬────────────┬──────────┐
│ month ┆ JB Line  ┆ Inline   ┆ Fogg      ┆ UNASSIGNED ┆ PD Line  │
│ ---   ┆ ---      ┆ ---      ┆ ---       ┆ ---        ┆ ---      │
│ i8    ┆ f64      ┆ f64      ┆ f64       ┆ f64        ┆ f64      │
╞═══════╪══════════╪══════════╪═══════════╪════════════╪══════════╡
│ 1     ┆ 65,530.0 ┆ 14,846.0 ┆ 228,660.0 ┆ 0.0        ┆ 26,893.0 │
│ 2     ┆ 74,278.0 ┆ 25,219.0 ┆ 128,070.0 ┆ 0.0        ┆ 40,678.0 │
│ 3     ┆ 64,967.0 ┆ 34,727.0 ┆ 232,188.0 ┆ 0.0        ┆ 30,950.0 │
│ 4     ┆ 61,366.0 ┆ 37,757.0 ┆ 207,246.0 ┆ 192.0      ┆ 38,618.0 │
│ 5     ┆ 76,314.0 ┆ 31,172.0 ┆ 186,972.0 ┆ 0.0        ┆ 30,474.0 │
│ 6     ┆ 52,650.0 ┆ 29,927.0 ┆ 261,276.0 ┆ 0.0        ┆ 33,729.0 │
│ 7     ┆ 22,734.0 ┆ 26,375.0 ┆ 560,478.0 ┆ 0.0        ┆ 31,658.0 │
│ 8     ┆ 14,560.0 ┆ 25,099.0 ┆ 526,416.0 ┆ 0.0        ┆ 16,118.0 │
│ 9     ┆ 44,522.0 ┆ 16,911.0 ┆ 648,678.0 ┆ 0.0        ┆ 14,532.0 │
│

In [14]:
# Weekly gallons by line
weekly_gal = (
    so_gal.lazy()
    .group_by("line", "week")
    .agg(pl.col("gallons").sum().round(0).alias("gal"))
    .collect()
    .pivot(on="line", index="week", values="gal")
    .fill_null(0)
    .sort("week")
)

print("=== Weekly Gallons by Line (first 12 weeks shown) ===")
with pl.Config(tbl_width_chars=200, tbl_rows=12, thousands_separator=","):
    print(weekly_gal.head(12))

# Weekly averages
n_weeks = weekly_gal.height
print(f"\n=== Weekly AVERAGE ({n_weeks} weeks) ===")
avg_row = {col: f"{weekly_gal[col].mean():,.0f}" for col in weekly_gal.columns if col != "week"}
for line, avg in sorted(avg_row.items(), key=lambda x: -float(x[1].replace(",",""))):
    print(f"  {line:12s}  {avg:>10s} gal/week")

# Grand summary
print(f"\n=== Overall Averages ===")
print(f"  {'':12s}  {'gal/week':>10s}  {'gal/month':>10s}  {'gal/year':>12s}")
for line in sorted(avg_row.keys(), key=lambda x: -float(avg_row[x].replace(",",""))):
    wk = weekly_gal[line].mean()
    mo = monthly_gal[line].mean()
    yr = line_totals.filter(pl.col("line") == line)["total_gal"][0]
    print(f"  {line:12s}  {wk:>10,.0f}  {mo:>10,.0f}  {yr:>12,.0f}")

=== Weekly Gallons by Line (first 12 weeks shown) ===
shape: (12, 6)
┌──────┬──────────┬──────────┬──────────┬───────────┬────────────┐
│ week ┆ PD Line  ┆ JB Line  ┆ Inline   ┆ Fogg      ┆ UNASSIGNED │
│ ---  ┆ ---      ┆ ---      ┆ ---      ┆ ---       ┆ ---        │
│ i8   ┆ f64      ┆ f64      ┆ f64      ┆ f64       ┆ f64        │
╞══════╪══════════╪══════════╪══════════╪═══════════╪════════════╡
│ 1    ┆ 7,062.0  ┆ 5,992.0  ┆ 4,754.0  ┆ 0.0       ┆ 0.0        │
│ 2    ┆ 10,260.0 ┆ 12,788.0 ┆ 3,230.0  ┆ 24,660.0  ┆ 0.0        │
│ 3    ┆ 5,638.0  ┆ 14,886.0 ┆ 5,001.0  ┆ 145,362.0 ┆ 0.0        │
│ 4    ┆ 2,190.0  ┆ 11,636.0 ┆ 3,939.0  ┆ 0.0       ┆ 0.0        │
│ 5    ┆ 6,618.0  ┆ 24,300.0 ┆ 490.0    ┆ 58,638.0  ┆ 0.0        │
│ 6    ┆ 7,563.0  ┆ 17,208.0 ┆ 4,149.0  ┆ 30,642.0  ┆ 0.0        │
│ 7    ┆ 10,606.0 ┆ 5,436.0  ┆ 12,917.0 ┆ 31,590.0  ┆ 0.0        │
│ 8    ┆ 9,835.0  ┆ 21,308.0 ┆ 2,993.0  ┆ 35,730.0  ┆ 0.0        │
│ 9    ┆ 12,674.0 ┆ 30,326.0 ┆ 5,160.0  ┆ 30,108.0  ┆ 0.0   

## 6c. Fogg Line (WW/AF) Capacity / Slack

Assumptions (edit as needed):
- Fogg instantaneous fill rate: **100 gal/min**
- Schedule: **4 days/week × 10 hr/day** (1 shift)
- Typical batch: **5,040 gal** (approx 840 cases of 6×1gal; 35-case pallets)

We convert shipped-equivalent gallons (from `SO` transactions) into:
- Fill time (value-add minutes at the nozzle)
- Batch / case / pallet equivalents
- An inferred **non-fill time per batch** (staging, changeovers, breaks, downtime) by looking at peak weeks.

In [15]:
# --- Fogg capacity assumptions ---
FOGG_GPM = 100.0

# Schedule inputs
FOGG_SHIFT_WINDOW_HOURS_PER_DAY = 11.0
FOGG_BREAK_MIN_PER_DAY = (2 * 15) + 30
FOGG_SHIFT_HOURS_PER_DAY = FOGG_SHIFT_WINDOW_HOURS_PER_DAY - (FOGG_BREAK_MIN_PER_DAY / 60.0)
FOGG_DAYS_PER_WEEK = 4.0

FOGG_SCHEDULED_HOURS_PER_WEEK = FOGG_SHIFT_HOURS_PER_DAY * FOGG_DAYS_PER_WEEK
FOGG_FILL_GAL_PER_HOUR = FOGG_GPM * 60.0

# Typical batch / order sizing
FOGG_BATCH_GAL = 5040.0
FOGG_CASES_PER_BATCH = 840.0
FOGG_CASES_PER_PALLET = 35.0

FOGG_GAL_PER_CASE_EQUIV = FOGG_BATCH_GAL / FOGG_CASES_PER_BATCH  # 6 gal/case (6x1gal)
FOGG_PALLETS_PER_BATCH = FOGG_CASES_PER_BATCH / FOGG_CASES_PER_PALLET  # 24
FOGG_FILL_MIN_PER_BATCH = FOGG_BATCH_GAL / FOGG_GPM  # 50.4 min

print(
    f"Fogg assumptions: {FOGG_GPM:.0f} gpm, "
    f"{FOGG_DAYS_PER_WEEK:.0f}x{FOGG_SHIFT_HOURS_PER_DAY:,.1f} hr/day "
    f"= {FOGG_SCHEDULED_HOURS_PER_WEEK:,.0f} hr/week"
)
print(f"Batch: {FOGG_BATCH_GAL:,.0f} gal = {FOGG_CASES_PER_BATCH:,.0f} cases = {FOGG_PALLETS_PER_BATCH:,.0f} pallets")
print(f"Fill time per batch @ {FOGG_GPM:.0f} gpm: {FOGG_FILL_MIN_PER_BATCH:.1f} min")


Fogg assumptions: 100 gpm, 4x10.0 hr/day = 40 hr/week
Batch: 5,040 gal = 840 cases = 24 pallets
Fill time per batch @ 100 gpm: 50.4 min


In [16]:
# --- Fogg weekly + monthly shipped-equivalent volume (2025) ---

fogg_weekly = (
    so_gal.lazy()
    .filter(pl.col("line") == "Fogg")
    .group_by(["yr", "week"])
    .agg(pl.col("gallons").sum().alias("gal"))
    .sort(["yr", "week"])
    .with_columns(
        (pl.col("gal") / FOGG_FILL_GAL_PER_HOUR).alias("fill_hours"),
        (pl.col("gal") / FOGG_BATCH_GAL).alias("batches"),
        (pl.col("gal") / FOGG_GAL_PER_CASE_EQUIV).alias("equiv_cases_6x1"),
    )
    .with_columns(
        (pl.col("equiv_cases_6x1") / FOGG_CASES_PER_PALLET).alias("equiv_pallets_35case"),
        (pl.col("fill_hours") / FOGG_DAYS_PER_WEEK).alias("fill_hours_per_day"),
        (pl.col("fill_hours") / FOGG_SCHEDULED_HOURS_PER_WEEK).alias("fill_utilization_of_schedule"),
        (pl.col("gal") / (FOGG_SCHEDULED_HOURS_PER_WEEK * 60)).alias("effective_gpm_if_full_40h"),
    )
    .collect()
)

base_total_gal = float(fogg_weekly["gal"].sum())
shipping_weeks = fogg_weekly.height

print(f"Fogg shipped-equivalent total (dataset year): {base_total_gal:,.0f} gal")
print(f"Weeks with any Fogg shipments in dataset: {shipping_weeks}")

# Summary stats over the *weeks with shipments* (not including zero weeks)
def _wk_stats(s: pl.Series) -> dict[str, float]:
    return {
        "mean": float(s.mean()),
        "median": float(s.median()),
        "p90": float(s.quantile(0.90)),
        "p95": float(s.quantile(0.95)),
        "max": float(s.max()),
    }

gal_stats = _wk_stats(fogg_weekly["gal"])
fill_hr_stats = _wk_stats(fogg_weekly["fill_hours"])

print("Fogg weekly gallons (weeks with shipments):")
print(f"  mean:   {gal_stats['mean']:,.0f}")
print(f"  median: {gal_stats['median']:,.0f}")
print(f"  p90:    {gal_stats['p90']:,.0f}")
print(f"  p95:    {gal_stats['p95']:,.0f}")
print(f"  max:    {gal_stats['max']:,.0f}")

print("Equivalent *fill time only* @ 100 gpm (weeks with shipments):")
print(f"  mean:   {fill_hr_stats['mean']:,.1f} hr/week  ({fill_hr_stats['mean']/FOGG_DAYS_PER_WEEK:,.1f} hr/day)")
print(f"  p95:    {fill_hr_stats['p95']:,.1f} hr/week  ({fill_hr_stats['p95']/FOGG_DAYS_PER_WEEK:,.1f} hr/day)")
print(f"  max:    {fill_hr_stats['max']:,.1f} hr/week  ({fill_hr_stats['max']/FOGG_DAYS_PER_WEEK:,.1f} hr/day)")

print("Top 10 Fogg weeks by gallons:")
with pl.Config(thousands_separator=",", tbl_width_chars=140):
    print(
        fogg_weekly
        .select(["yr", "week", "gal", "batches", "fill_hours", "fill_hours_per_day", "effective_gpm_if_full_40h"])
        .sort("gal", descending=True)
        .head(10)
    )

# Monthly view (busy-season shape)
fogg_monthly = (
    so_gal.lazy()
    .filter(pl.col("line") == "Fogg")
    .group_by(["yr", "month"])
    .agg(pl.col("gallons").sum().alias("gal"))
    .sort(["yr", "month"])
    .with_columns(
        (pl.col("gal") / FOGG_FILL_GAL_PER_HOUR).alias("fill_hours"),
        (pl.col("gal") / FOGG_BATCH_GAL).alias("batches"),
        (pl.col("gal") / FOGG_GAL_PER_CASE_EQUIV).alias("equiv_cases_6x1"),
    )
    .with_columns(
        (pl.col("equiv_cases_6x1") / FOGG_CASES_PER_PALLET).alias("equiv_pallets_35case"),
    )
    .collect()
)

print("Fogg monthly shipped-equivalent gallons:")
with pl.Config(thousands_separator=",", tbl_width_chars=120, tbl_rows=fogg_monthly.height):
    print(fogg_monthly)


Fogg shipped-equivalent total (dataset year): 4,144,086 gal
Weeks with any Fogg shipments in dataset: 48
Fogg weekly gallons (weeks with shipments):
  mean:   86,335
  median: 70,689
  p90:    151,926
  p95:    157,446
  max:    160,536
Equivalent *fill time only* @ 100 gpm (weeks with shipments):
  mean:   14.4 hr/week  (3.6 hr/day)
  p95:    26.2 hr/week  (6.6 hr/day)
  max:    26.8 hr/week  (6.7 hr/day)
Top 10 Fogg weeks by gallons:
shape: (10, 7)
┌───────┬──────┬───────────┬───────────┬────────────┬────────────────────┬───────────────────────────┐
│ yr    ┆ week ┆ gal       ┆ batches   ┆ fill_hours ┆ fill_hours_per_day ┆ effective_gpm_if_full_40h │
│ ---   ┆ ---  ┆ ---       ┆ ---       ┆ ---        ┆ ---                ┆ ---                       │
│ i32   ┆ i8   ┆ f64       ┆ f64       ┆ f64        ┆ f64                ┆ f64                       │
╞═══════╪══════╪═══════════╪═══════════╪════════════╪════════════════════╪═══════════════════════════╡
│ 2,025 ┆ 38   ┆ 160,536.0 ┆ 3

In [17]:
# --- The trick: infer non-fill minutes per batch from peak weeks ---
#
# 100 gpm is an instantaneous fill rate. Over a full shift/week, you also have
# staging, changeovers, label changes, breaks, QC holds, forklift time, etc.
#
# If we assume the top-volume weeks represent "running hard" on the 40-hr schedule,
# then (40hr - fill_hours) approximates non-fill time. Divide by batches to infer
# non-fill minutes per batch.

TOP_WEEKS_FOR_OVERHEAD = 12

peak_weeks = (
    fogg_weekly
    .sort("gal", descending=True)
    .head(TOP_WEEKS_FOR_OVERHEAD)
    .with_columns(
        (pl.lit(FOGG_SCHEDULED_HOURS_PER_WEEK) - pl.col("fill_hours")).alias("non_fill_hours_if_40h"),
        ((pl.lit(FOGG_SCHEDULED_HOURS_PER_WEEK) - pl.col("fill_hours")) * 60 / pl.col("batches")).alias("non_fill_min_per_batch_if_40h"),
    )
)

nf = peak_weeks["non_fill_min_per_batch_if_40h"]

nf_median = float(nf.median())
nf_p25 = float(nf.quantile(0.25))
nf_p75 = float(nf.quantile(0.75))

total_min_per_batch_median = FOGG_FILL_MIN_PER_BATCH + nf_median

print(f"Using top {TOP_WEEKS_FOR_OVERHEAD} weeks as 'peak' sample:")
print(f"  inferred non-fill minutes per 5,040-gal batch (median): {nf_median:,.1f} min")
print(f"  (p25={nf_p25:,.1f}, p75={nf_p75:,.1f})")
print(f"  fill minutes per batch @ 100 gpm: {FOGG_FILL_MIN_PER_BATCH:,.1f} min")
print(f"  total minutes per batch (fill + non-fill, median): {total_min_per_batch_median:,.1f} min")

# Convert that into an "effective" weekly capacity on the 40-hr schedule
sched_min_week = FOGG_SCHEDULED_HOURS_PER_WEEK * 60
batches_per_week_eff = sched_min_week / total_min_per_batch_median
cap_gal_per_week_eff = batches_per_week_eff * FOGG_BATCH_GAL
cap_gpm_eff = cap_gal_per_week_eff / sched_min_week

print("Implied effective capacity (if you can sustain the median peak-week non-fill/batch):")
print(f"  {cap_gal_per_week_eff:,.0f} gal/week on a {FOGG_SCHEDULED_HOURS_PER_WEEK:.0f} hr/week schedule")
print(f"  effective rate: {cap_gpm_eff:,.1f} gpm over the scheduled week")

# What do common weekly targets translate to in scheduled hours?
WEEKLY_TARGET_GAL = 100_000.0  # per comms: "can handle 100k gal/week today" (edit if this is incremental vs total)

targets = [
    ("100k/week", WEEKLY_TARGET_GAL),
    ("p95 shipped week", float(fogg_weekly["gal"].quantile(0.95))),
    ("max shipped week", float(fogg_weekly["gal"].max())),
]

print("Weekly targets translated into time (using median minutes/batch):")
for label, gal in targets:
    fill_hr = gal / FOGG_FILL_GAL_PER_HOUR
    batches = gal / FOGG_BATCH_GAL
    total_hr = batches * total_min_per_batch_median / 60
    slack_hr = FOGG_SCHEDULED_HOURS_PER_WEEK - total_hr
    print(
        f"  {label:16s}: {gal:9,.0f} gal  "
        f"fill={fill_hr:5.1f}h  total={total_hr:5.1f}h  slack_vs_40h={slack_hr:5.1f}h"
    )


# Translate 2025 volume into batch-time hours using the inferred total minutes/batch
base_batches = base_total_gal / FOGG_BATCH_GAL
base_batch_time_hours = base_batches * total_min_per_batch_median / 60

scheduled_hours_year = 52 * FOGG_SCHEDULED_HOURS_PER_WEEK
scheduled_days_year = 52 * FOGG_DAYS_PER_WEEK

print("2025 Fogg volume translated into time (using inferred total minutes/batch):")
print(f"  total batches: {base_batches:,.1f}")
print(f"  total batch-time: {base_batch_time_hours:,.0f} hours")
print(f"  avg batch-time per scheduled day (4-day weeks): {base_batch_time_hours/scheduled_days_year:,.1f} hr/day")
print(f"  avg batch-time per scheduled week: {base_batch_time_hours/52:,.1f} hr/week")
print(f"  scheduled hours/year (1 shift): {scheduled_hours_year:,.0f} hours")

Using top 12 weeks as 'peak' sample:
  inferred non-fill minutes per 5,040-gal batch (median): 29.3 min
  (p25=28.7, p75=31.7)
  fill minutes per batch @ 100 gpm: 50.4 min
  total minutes per batch (fill + non-fill, median): 79.7 min
Implied effective capacity (if you can sustain the median peak-week non-fill/batch):
  151,791 gal/week on a 40 hr/week schedule
  effective rate: 63.2 gpm over the scheduled week
Weekly targets translated into time (using median minutes/batch):
  100k/week       :   100,000 gal  fill= 16.7h  total= 26.4h  slack_vs_40h= 13.6h
  p95 shipped week:   157,446 gal  fill= 26.2h  total= 41.5h  slack_vs_40h= -1.5h
  max shipped week:   160,536 gal  fill= 26.8h  total= 42.3h  slack_vs_40h= -2.3h
2025 Fogg volume translated into time (using inferred total minutes/batch):
  total batches: 822.2
  total batch-time: 1,092 hours
  avg batch-time per scheduled day (4-day weeks): 5.3 hr/day
  avg batch-time per scheduled week: 21.0 hr/week
  scheduled hours/year (1 shift)

In [18]:
# --- Peak-season check: overhead inference for Aug-Oct dominant weeks ---
# We classify each ISO week by the month that contains the most Fogg gallons (dominant month).
# Then we look at the highest-volume weeks in months 8-10 to see whether the inferred
# non-fill minutes per 5,040-gal batch is consistent with the overall peak-week estimate.

PEAK_MONTHS = [8, 9, 10]
PEAK_WEEK_GAL_MIN = 140_000.0  # adjust as desired

# Dominant month per (yr, week)
fogg_wk_month = (
    so_gal.lazy()
    .filter(pl.col("line") == "Fogg")
    .group_by(["yr", "week", "month"])
    .agg(pl.col("gallons").sum().alias("gal_month"))
    .collect()
    .sort(["yr", "week", "gal_month"], descending=[False, False, True])
)

fogg_wk_dom_month = (
    fogg_wk_month
    .group_by(["yr", "week"])
    .agg(pl.first("month").alias("dom_month"))
)

fogg_wk = (
    fogg_weekly
    .join(fogg_wk_dom_month, on=["yr", "week"], how="left")
    .with_columns(
        ((pl.lit(FOGG_SCHEDULED_HOURS_PER_WEEK) - pl.col("fill_hours")) * 60 / pl.col("batches")).alias(
            "non_fill_min_per_batch_if_40h"
        ),
    )
)

wk_810 = fogg_wk.filter(pl.col("dom_month").is_in(PEAK_MONTHS))

wk_810_peak = wk_810.filter(pl.col("gal") >= pl.lit(PEAK_WEEK_GAL_MIN))

print(f"Weeks with dominant month in {PEAK_MONTHS}: {wk_810.height}")
print(f"Weeks in {PEAK_MONTHS} with >= {PEAK_WEEK_GAL_MIN:,.0f} gal: {wk_810_peak.height}")

s = wk_810_peak["non_fill_min_per_batch_if_40h"]

if wk_810_peak.height:
    print("\nInferred non-fill minutes per 5,040-gal batch (peak weeks, months 8-10):")
    print(f"  min:    {float(s.min()):,.1f}")
    print(f"  p25:    {float(s.quantile(0.25)):,.1f}")
    print(f"  median: {float(s.median()):,.1f}")
    print(f"  p75:    {float(s.quantile(0.75)):,.1f}")
    print(f"  max:    {float(s.max()):,.1f}")

    print("\nPeak weeks detail:")
    with pl.Config(thousands_separator=",", tbl_width_chars=160, tbl_rows=30):
        print(
            wk_810_peak
            .select(["yr", "week", "dom_month", "gal", "batches", "fill_hours", "non_fill_min_per_batch_if_40h"])
            .sort("gal", descending=True)
        )
else:
    print("No peak weeks found for the chosen threshold.")


Weeks with dominant month in [8, 9, 10]: 13
Weeks in [8, 9, 10] with >= 140,000 gal: 8

Inferred non-fill minutes per 5,040-gal batch (peak weeks, months 8-10):
  min:    24.9
  p25:    26.4
  median: 28.9
  p75:    29.4
  max:    32.5

Peak weeks detail:
shape: (8, 7)
┌───────┬──────┬───────────┬───────────┬───────────┬────────────┬───────────────────────────────┐
│ yr    ┆ week ┆ dom_month ┆ gal       ┆ batches   ┆ fill_hours ┆ non_fill_min_per_batch_if_40h │
│ ---   ┆ ---  ┆ ---       ┆ ---       ┆ ---       ┆ ---        ┆ ---                           │
│ i32   ┆ i8   ┆ i8        ┆ f64       ┆ f64       ┆ f64        ┆ f64                           │
╞═══════╪══════╪═══════════╪═══════════╪═══════════╪════════════╪═══════════════════════════════╡
│ 2,025 ┆ 38   ┆ 9         ┆ 160,536.0 ┆ 31.852381 ┆ 26.756     ┆ 24.947586                     │
│ 2,025 ┆ 41   ┆ 10        ┆ 158,214.0 ┆ 31.391667 ┆ 26.369     ┆ 26.053411                     │
│ 2,025 ┆ 37   ┆ 9         ┆ 157,446.0 ┆ 31.

In [19]:
# --- Increment scenario: +4,000,000 gal/year WW/AF on Fogg ---

INCREMENTAL_GAL_PER_YEAR = 4_000_000.0

inc_batches = INCREMENTAL_GAL_PER_YEAR / FOGG_BATCH_GAL
inc_batch_time_hours = inc_batches * total_min_per_batch_median / 60

print(f"Increment: +{INCREMENTAL_GAL_PER_YEAR:,.0f} gal/year")
print(f"  +{inc_batches:,.1f} batches/year")
print(f"  +{inc_batch_time_hours:,.0f} batch-time hours/year (using median peak-week minutes/batch)")
print(f"  avg +{inc_batch_time_hours/52:,.1f} hr/week | +{inc_batch_time_hours/(52*FOGG_DAYS_PER_WEEK):,.1f} hr/day")

# Scenario A: incremental volume is spread uniformly across the year
inc_weekly_uniform = INCREMENTAL_GAL_PER_YEAR / 52

scenario_uniform = (
    fogg_weekly
    .with_columns(
        (pl.col("gal") + pl.lit(inc_weekly_uniform)).alias("gal_plus_inc_uniform"),
    )
)

# Scenario B: incremental volume follows the same weekly seasonality as 2025 shipments (scaled)
mult = 1.0 + (INCREMENTAL_GAL_PER_YEAR / base_total_gal)
scenario_scaled = fogg_weekly.with_columns((pl.col("gal") * pl.lit(mult)).alias("gal_plus_inc_scaled"))

# Convert weekly gallons to required batch-time hours using inferred minutes/batch
# (This assumes the same batch sizing and non-fill/batch holds at higher volumes.)
def _add_req_cols(df: pl.DataFrame, gal_col: str) -> pl.DataFrame:
    batches_expr = (pl.col(gal_col) / FOGG_BATCH_GAL)
    hours_req_expr = (batches_expr * pl.lit(total_min_per_batch_median) / 60)
    hours_over_expr = (hours_req_expr - pl.lit(FOGG_SCHEDULED_HOURS_PER_WEEK))

    # Polars evaluates with_columns expressions in parallel; avoid referencing newly created columns.
    return df.with_columns(
        batches_expr.alias("batches_req"),
        hours_req_expr.alias("hours_req"),
        hours_over_expr.alias("hours_over_40"),
        (hours_over_expr / pl.lit(FOGG_SHIFT_HOURS_PER_DAY)).alias("extra_10hr_days"),
    )
u = _add_req_cols(scenario_uniform, "gal_plus_inc_uniform")
s = _add_req_cols(scenario_scaled, "gal_plus_inc_scaled")

print("Uniform increment: worst weeks vs 40hr schedule")
with pl.Config(thousands_separator=",", tbl_width_chars=140):
    print(
        u
        .select(["yr","week","gal_plus_inc_uniform","hours_req","hours_over_40","extra_10hr_days"])
        .sort("hours_req", descending=True)
        .head(8)
    )

print("Seasonality-scaled increment: worst weeks vs 40hr schedule")
with pl.Config(thousands_separator=",", tbl_width_chars=140):
    print(
        s
        .select(["yr","week","gal_plus_inc_scaled","hours_req","hours_over_40","extra_10hr_days"])
        .sort("hours_req", descending=True)
        .head(8)
    )

print("Interpretation notes:")
print("  - If hours_over_40 is positive, you'd need overtime / extra day(s) *unless* you prebuild inventory in earlier weeks.")
print("  - The seasonality-scaled case is the 'worst' if the new demand peaks at the same time as existing WW/AF demand.")

Increment: +4,000,000 gal/year
  +793.7 batches/year
  +1,054 batch-time hours/year (using median peak-week minutes/batch)
  avg +20.3 hr/week | +5.1 hr/day
Uniform increment: worst weeks vs 40hr schedule
shape: (8, 6)
┌───────┬──────┬──────────────────────┬───────────┬───────────────┬─────────────────┐
│ yr    ┆ week ┆ gal_plus_inc_uniform ┆ hours_req ┆ hours_over_40 ┆ extra_10hr_days │
│ ---   ┆ ---  ┆ ---                  ┆ ---       ┆ ---           ┆ ---             │
│ i32   ┆ i8   ┆ f64                  ┆ f64       ┆ f64           ┆ f64             │
╞═══════╪══════╪══════════════════════╪═══════════╪═══════════════╪═════════════════╡
│ 2,025 ┆ 38   ┆ 237,459.076923       ┆ 62.575321 ┆ 22.575321     ┆ 2.257532        │
│ 2,025 ┆ 41   ┆ 235,137.076923       ┆ 61.963427 ┆ 21.963427     ┆ 2.196343        │
│ 2,025 ┆ 37   ┆ 234,369.076923       ┆ 61.761043 ┆ 21.761043     ┆ 2.176104        │
│ 2,025 ┆ 33   ┆ 229,893.076923       ┆ 60.581526 ┆ 20.581526     ┆ 2.058153        │
│ 2,025

In [20]:
# --- Optional: simple weekly leveling model (inventory build) ---
# Treat weekly shipments as demand. Assume you can produce up to CAP_GAL_PER_WEEK each week.
# If you always produce at capacity, inventory evolves as:
#   inv[t] = inv[t-1] + CAP - demand[t]
# Starting inventory needed to avoid stockouts is: max(0, -min(cumsum(CAP - demand))).

# Build a full 52-week calendar (weeks 1..52) for the dataset year.
y = int(fogg_weekly["yr"][0])
cal = pl.DataFrame({"yr": [y]*52, "week": list(range(1, 53))})

weekly_demand = (
    cal
    .join(fogg_weekly.select(["yr","week","gal"]), on=["yr","week"], how="left")
    .with_columns(pl.col("gal").fill_null(0.0).alias("demand_gal"))
)

CAP_GAL_PER_WEEK = float(gal_stats["p95"])  # p95 shipped week as a conservative 40-hr capacity proxy
print(f"Assumed CAP_GAL_PER_WEEK: {CAP_GAL_PER_WEEK:,.0f} gal/week")

# Two demand scenarios
weekly_demand = weekly_demand.with_columns(
    (pl.col("demand_gal") + pl.lit(INCREMENTAL_GAL_PER_YEAR/52)).alias("demand_uniform"),
    (pl.col("demand_gal") * pl.lit(mult)).alias("demand_scaled"),
)

def leveling_report(demand_col: str):
    tmp = (
        weekly_demand
        .select([
            "week",
            pl.col(demand_col).alias("demand"),
            (pl.lit(CAP_GAL_PER_WEEK) - pl.col(demand_col)).alias("delta"),
        ])
        .with_columns(pl.col("delta").cum_sum().alias("cum_delta"))
    )

    min_cum = float(tmp["cum_delta"].min())
    start_inv = -min_cum if min_cum < 0 else 0.0
    tmp = tmp.with_columns((pl.col("cum_delta") + pl.lit(start_inv)).alias("inv"))

    peak_inv = float(tmp["inv"].max())
    end_inv = float(tmp["inv"][-1])

    return start_inv, peak_inv, end_inv, tmp

for name, col in [("uniform", "demand_uniform"), ("scaled", "demand_scaled")]:
    start, peak, end, tmp = leveling_report(col)
    print(f"Leveling scenario: {name}")
    print(f"  total demand: {float(tmp['demand'].sum()):,.0f} gal")
    print(f"  start inventory needed (to avoid any stockout): {start:,.0f} gal")
    print(f"  peak inventory during year (if you run at CAP every week): {peak:,.0f} gal")
    print(f"  end inventory: {end:,.0f} gal")

    # Convert peak inventory to batches/pallets for storage discussion
    peak_batches = peak / FOGG_BATCH_GAL
    peak_pallets = peak_batches * FOGG_PALLETS_PER_BATCH
    print(f"  peak inventory approx {peak_batches:,.0f} batches approx {peak_pallets:,.0f} pallets (35-case)")

# Show the worst drawdown / build weeks for the scaled case
start, peak, end, tmp_scaled = leveling_report('demand_scaled')
print("Scaled case: weeks with highest inventory (top 8)")
with pl.Config(thousands_separator=",", tbl_width_chars=120):
    print(tmp_scaled.sort('inv', descending=True).head(8))

Assumed CAP_GAL_PER_WEEK: 157,446 gal/week
Leveling scenario: uniform
  total demand: 8,144,086 gal
  start inventory needed (to avoid any stockout): 37,417 gal
  peak inventory during year (if you run at CAP every week): 967,124 gal
  end inventory: 80,523 gal
  peak inventory approx 192 batches approx 4,605 pallets (35-case)
Leveling scenario: scaled
  total demand: 8,144,086 gal
  start inventory needed (to avoid any stockout): 114,340 gal
  peak inventory during year (if you run at CAP every week): 1,919,825 gal
  end inventory: 157,446 gal
  peak inventory approx 381 batches approx 9,142 pallets (35-case)
Scaled case: weeks with highest inventory (top 8)
shape: (8, 5)
┌──────┬────────────────┬─────────────────┬──────────────────┬──────────────────┐
│ week ┆ demand         ┆ delta           ┆ cum_delta        ┆ inv              │
│ ---  ┆ ---            ┆ ---             ┆ ---              ┆ ---              │
│ i64  ┆ f64            ┆ f64             ┆ f64              ┆ f64      

### Bottle Supply Constraint (Fogg 1-gal Bottle)

From the 2025 SO-derived dataset, **all Fogg (WW/AF) volume uses a single 1-gallon bottle**:
- `BOTTLE-RIGS NATURAL  120gm` (128 oz)

If that bottle is produced in-house, it can become a real constraint (especially in the busy season) because:
- Fogg filler consumes **100 bottles/min** while running (at 100 gpm, 1 gal/bottle)
- Bottle output is spread across the bottle schedule (often 24/7), while filling is concentrated into a 4×10 schedule

The analysis below converts shipments into **bottles/week** and compares against **actual BR receipts** (supply-side) for the same bottle.


In [21]:
# --- Bottle demand for the Fogg 1-gal bottle (and how concentrated it is) ---

RIGS_BOTTLE_DESC = "BOTTLE-RIGS NATURAL  120gm"

# Bottles required per shipped transaction:
# abs(transactionqty) * qtyperbill == bottles (for 1-gal bottles, this equals gallons)
rigs_txn = (
    so_gal.lazy()
    .with_columns(
        (pl.col("transactionqty").abs() * pl.col("qtyperbill")).alias("bottles"),
    )
    .filter(pl.col("Bottle") == RIGS_BOTTLE_DESC)
    .collect()
)

print("RIGS bottle rows:", rigs_txn.height)
print("Lines using this bottle:")
with pl.Config(thousands_separator=",", tbl_width_chars=120):
    print(
        rigs_txn
        .group_by("line")
        .agg(pl.col("bottles").sum().alias("bottles"))
        .sort("bottles", descending=True)
    )

# Confirm Fogg uses exactly one bottle type
fogg_bottle_types = (
    so_gal
    .filter(pl.col("line") == "Fogg")
    .select(["Bottle", "bottle_oz"])
    .unique()
)
print("Fogg unique bottle types:")
print(fogg_bottle_types)

rigs_weekly = (
    rigs_txn.lazy()
    .group_by(["yr", "week", "line"])
    .agg(pl.col("bottles").sum().alias("bottles"))
    .collect()
)

# Total weekly bottles for this bottle across all lines
rigs_weekly_total = (
    rigs_weekly
    .group_by(["yr", "week"])
    .agg(pl.col("bottles").sum().alias("bottles"))
    .sort(["yr", "week"])
)

# Fogg-only weekly bottles (dominant)
rigs_weekly_fogg = (
    rigs_weekly
    .filter(pl.col("line") == "Fogg")
    .group_by(["yr", "week"])
    .agg(pl.col("bottles").sum().alias("bottles"))
    .sort(["yr", "week"])
)

base_bottles_year = float(rigs_txn.filter(pl.col("line") == "Fogg")["bottles"].sum())
base_bottles_total = float(rigs_txn["bottles"].sum())

print(f"2025 Fogg bottles (this bottle): {base_bottles_year:,.0f}")
print(f"2025 total bottles (this bottle, all lines): {base_bottles_total:,.0f}")

print("Top 10 weeks (Fogg-only) by bottles:")
with pl.Config(thousands_separator=",", tbl_width_chars=120):
    print(rigs_weekly_fogg.sort("bottles", descending=True).head(10))

RIGS bottle rows: 759
Lines using this bottle:
shape: (2, 2)
┌─────────┬────────────┐
│ line    ┆ bottles    │
│ ---     ┆ ---        │
│ str     ┆ f64        │
╞═════════╪════════════╡
│ Fogg    ┆ 4.144086e6 │
│ JB Line ┆ 97,336.0   │
└─────────┴────────────┘
Fogg unique bottle types:
shape: (1, 2)
┌────────────────────────────┬───────────┐
│ Bottle                     ┆ bottle_oz │
│ ---                        ┆ ---       │
│ str                        ┆ f64       │
╞════════════════════════════╪═══════════╡
│ BOTTLE-RIGS NATURAL  120gm ┆ 128.0     │
└────────────────────────────┴───────────┘
2025 Fogg bottles (this bottle): 4,144,086
2025 total bottles (this bottle, all lines): 4,241,422
Top 10 weeks (Fogg-only) by bottles:
shape: (10, 3)
┌───────┬──────┬───────────┐
│ yr    ┆ week ┆ bottles   │
│ ---   ┆ ---  ┆ ---       │
│ i32   ┆ i8   ┆ f64       │
╞═══════╪══════╪═══════════╡
│ 2,025 ┆ 38   ┆ 160,536.0 │
│ 2,025 ┆ 41   ┆ 158,214.0 │
│ 2,025 ┆ 37   ┆ 157,446.0 │
│ 2,025 ┆ 33   ┆

In [22]:
# --- Compare bottle demand to bottle *supply* (actual BR receipts) ---
# Supply comes from IM receipts where TransactionCode='BR' for item 022001.
# Demand comes from SO shipments that use this bottle.

# Parse BR receipt dates and aggregate supply
br = (
    br_txns.lazy()
    .with_columns(
        pl.col("transactiondate_iso").str.strptime(pl.Date, "%Y-%m-%d").alias("date"),
    )
    .with_columns(
        pl.col("date").dt.month().alias("month"),
        pl.col("date").dt.iso_year().alias("yr"),
        pl.col("date").dt.week().alias("week"),
    )
    .collect()
)

br_daily = (
    br.lazy()
    .group_by(["yr", "date"])
    .agg(pl.col("bottles").sum().alias("br_bottles"))
    .sort(["yr", "date"])
    .collect()
)

br_weekly = (
    br.lazy()
    .group_by(["yr", "week"])
    .agg(pl.col("bottles").sum().alias("br_bottles"))
    .sort(["yr", "week"])
    .collect()
)

br_monthly = (
    br.lazy()
    .group_by(["yr", "month"])
    .agg(pl.col("bottles").sum().alias("br_bottles"))
    .sort(["yr", "month"])
    .collect()
)

# Basic supply stats (helps validate "~30k/day" type assumptions)
br_s = br_daily["br_bottles"]

total_br = float(br_s.sum())
days_with_receipts = br_daily.height

print("Bottle BR receipts (022001, dataset year):")
print(f"  total receipts: {total_br:,.0f} bottles")
print(f"  days with receipts: {days_with_receipts}")
print(
    "  per receipt-day: "
    f"avg={float(br_s.mean()):,.0f} | p50={float(br_s.median()):,.0f} | p95={float(br_s.quantile(0.95)):,.0f} | max={float(br_s.max()):,.0f}"
)

# Weekly demand for this bottle (already computed in the previous cell)
rigs_monthly_total = (
    rigs_txn.lazy()
    .group_by(["yr", "month"])
    .agg(pl.col("bottles").sum().alias("bottles"))
    .sort(["yr", "month"])
    .collect()
)

# Build a week key from the union of demand weeks and supply weeks
wk_key = (
    pl.concat(
        [
            rigs_weekly_total.select(["yr", "week"]),
            br_weekly.select(["yr", "week"]),
        ],
        how="vertical",
    )
    .unique()
    .sort(["yr", "week"])
)

bottle_weekly = (
    wk_key
    .join(rigs_weekly_total.rename({"bottles": "demand_total"}), on=["yr", "week"], how="left")
    .join(rigs_weekly_fogg.rename({"bottles": "demand_fogg"}), on=["yr", "week"], how="left")
    .join(br_weekly.rename({"br_bottles": "supply_br"}), on=["yr", "week"], how="left")
    .with_columns(
        pl.col("demand_total").fill_null(0.0),
        pl.col("demand_fogg").fill_null(0.0),
        pl.col("supply_br").fill_null(0.0),
    )
    .with_columns(
        (pl.col("demand_total") - pl.col("demand_fogg")).alias("demand_other"),
    )
)

print("Weekly totals (bottles):")
print(f"  demand (all lines): {float(bottle_weekly['demand_total'].sum()):,.0f}")
print(f"  demand (Fogg only): {float(bottle_weekly['demand_fogg'].sum()):,.0f}")
print(f"  supply (BR receipts): {float(bottle_weekly['supply_br'].sum()):,.0f}")

print("Peak weekly demand (bottles) for this bottle:")
print(f"  total max: {float(bottle_weekly['demand_total'].max()):,.0f}/week")
print(f"  Fogg max:  {float(bottle_weekly['demand_fogg'].max()):,.0f}/week")


def _buffer_variable(demand: list[float], supply: list[float]) -> list[float]:
    """Required inventory entering each week to avoid any stockout through the end of the horizon."""
    inv_next = 0.0
    buf = [0.0] * len(demand)
    for i in range(len(demand) - 1, -1, -1):
        inv_i = inv_next + demand[i] - supply[i]
        if inv_i < 0:
            inv_i = 0.0
        buf[i] = inv_i
        inv_next = inv_i
    return buf


def _report_supply_vs_demand(df: pl.DataFrame, demand_col: str, supply_col: str):
    demand = [float(x) for x in df[demand_col].to_list()]
    supply = [float(x) for x in df[supply_col].to_list()]

    deficits = [d - s for d, s in zip(demand, supply)]
    weeks_over = sum(1 for x in deficits if x > 0)
    worst_def = max(deficits) if deficits else 0.0

    buf = _buffer_variable(demand, supply)
    peak_buf = max(buf) if buf else 0.0
    peak_week = int(df["week"][buf.index(peak_buf)]) if buf else None

    start_buf = buf[0] if buf else 0.0

    print(f"{demand_col} vs {supply_col}:")
    print(f"  weeks demand > supply: {weeks_over}")
    print(f"  worst week deficit: {max(worst_def, 0.0):,.0f} bottles")
    print(f"  start buffer needed: {start_buf:,.0f} bottles")
    if peak_week is not None:
        print(f"  peak buffer needed:  {peak_buf:,.0f} bottles (entering week {peak_week})")

    return buf


# Scenario demand: baseline + incremental WW/AF gallons (1 gal = 1 bottle)
inc_year = INCREMENTAL_GAL_PER_YEAR if "INCREMENTAL_GAL_PER_YEAR" in globals() else 4_000_000.0
inc_week_uniform = inc_year / 52.0

# Scale only the Fogg portion, keeping other-line usage of this bottle unchanged
mult_inc = 1.0 + (inc_year / base_total_gal)

bottle_weekly = bottle_weekly.with_columns(
    pl.col("demand_total").alias("demand_baseline"),
    (pl.col("demand_total") + pl.lit(inc_week_uniform)).alias("demand_uniform"),
    (pl.col("demand_other") + (pl.col("demand_fogg") * pl.lit(mult_inc))).alias("demand_scaled"),
)

print(f"Scenario increment: +{inc_year:,.0f} gal/year (treated as +bottles/year)")

buf_baseline = _report_supply_vs_demand(bottle_weekly, "demand_baseline", "supply_br")
buf_uniform = _report_supply_vs_demand(bottle_weekly, "demand_uniform", "supply_br")
buf_scaled = _report_supply_vs_demand(bottle_weekly, "demand_scaled", "supply_br")

# Add buffers to the table for export
bottle_weekly = bottle_weekly.with_columns(
    pl.Series("buffer_baseline", buf_baseline),
    pl.Series("buffer_uniform", buf_uniform),
    pl.Series("buffer_scaled", buf_scaled),
)

# A compact multi-increment summary (helps with bid-volume sensitivity)
INCREMENT_OPTIONS = [
    0.0,
    4_000_000.0,
    5_500_000.0,
    7_300_000.0,
]

rows = []
for inc in INCREMENT_OPTIONS:
    inc_wk = inc / 52.0
    mult_i = 1.0 + (inc / base_total_gal)

    tmp = bottle_weekly.select([
        "yr",
        "week",
        "supply_br",
        pl.col("demand_total").alias("demand_baseline"),
        (pl.col("demand_total") + pl.lit(inc_wk)).alias("demand_uniform"),
        (pl.col("demand_other") + (pl.col("demand_fogg") * pl.lit(mult_i))).alias("demand_scaled"),
    ])

    # Baseline demand is the same regardless of the increment volume; only compute it once.
    modes = ["demand_baseline"] if inc == 0.0 else ["demand_uniform", "demand_scaled"]

    for mode in modes:
        buf = _buffer_variable(
            [float(x) for x in tmp[mode].to_list()],
            [float(x) for x in tmp["supply_br"].to_list()],
        )
        peak_buf = max(buf) if buf else 0.0
        peak_week = int(tmp["week"][buf.index(peak_buf)]) if buf else None

        deficits = [float(d) - float(s) for d, s in zip(tmp[mode].to_list(), tmp["supply_br"].to_list())]
        weeks_over = sum(1 for x in deficits if x > 0)
        worst_def = max(deficits) if deficits else 0.0

        rows.append(
            {
                "increment_gal_year": inc,
                "mode": mode.replace("demand_", ""),
                "weeks_over": weeks_over,
                "worst_week_deficit_bottles": max(worst_def, 0.0),
                "peak_buffer_bottles": peak_buf,
                "peak_buffer_week": peak_week,
            }
        )

bottle_scenario_summary = pl.DataFrame(rows).sort(["increment_gal_year", "mode"])

print("Bottle feasibility summary (vs 2025 BR receipts supply):")
with pl.Config(thousands_separator=",", tbl_width_chars=140, tbl_rows=bottle_scenario_summary.height):
    print(bottle_scenario_summary)


Bottle BR receipts (022001, dataset year):


  total receipts: 4,304,650 bottles
  days with receipts: 172
  per receipt-day: avg=25,027 | p50=26,353 | p95=41,724 | max=79,332
Weekly totals (bottles):
  demand (all lines): 4,241,422
  demand (Fogg only): 4,144,086
  supply (BR receipts): 4,304,650
Peak weekly demand (bottles) for this bottle:
  total max: 160,536/week
  Fogg max:  160,536/week
Scenario increment: +4,000,000 gal/year (treated as +bottles/year)
demand_baseline vs supply_br:
  weeks demand > supply: 24
  worst week deficit: 66,413 bottles
  start buffer needed: 26,806 bottles
  peak buffer needed:  146,489 bottles (entering week 17)
demand_uniform vs supply_br:
  weeks demand > supply: 51
  worst week deficit: 143,336 bottles
  start buffer needed: 3,936,772 bottles
  peak buffer needed:  3,936,772 bottles (entering week 2)
demand_scaled vs supply_br:
  weeks demand > supply: 46
  worst week deficit: 203,885 bottles
  start buffer needed: 3,978,695 bottles
  peak buffer needed:  3,979,720 bottles (entering week 3)


### Meeting Talking Points (Auto-Generated)

This cell prints a short, numbers-first summary based on the assumptions above (SO-based demand proxy + inferred overhead + bottle supply assumptions).

In [23]:
# Meeting cheat sheet

# Fogg demand stats
fogg_total = float(so_gal.filter(pl.col('line') == 'Fogg')['gallons'].sum())
fogg_p95 = float(fogg_weekly['gal'].quantile(0.95))
fogg_max = float(fogg_weekly['gal'].max())

print('Fogg (WW/AF) demand proxy (SO shipments, dataset year):')
print(f'  total: {fogg_total:,.0f} gal')
print(f'  p95 week: {fogg_p95:,.0f} gal/week | max week: {fogg_max:,.0f} gal/week')

print('Fogg effective throughput (from peak-week overhead inference):')
print(f'  non-fill minutes per 5,040-gal batch (median): {nf_median:,.1f} min')
print(f'  effective rate over schedule: {cap_gpm_eff:,.1f} gpm')
print(f'  implied 40-hr capacity: {cap_gal_per_week_eff:,.0f} gal/week')

# 100k/week statement
weekly_target = 100_000.0
fill_hr = weekly_target / FOGG_FILL_GAL_PER_HOUR
batches = weekly_target / FOGG_BATCH_GAL
total_hr = batches * total_min_per_batch_median / 60
print('Rule-of-thumb for 100k gal/week on Fogg (using inferred minutes/batch):')
print(f'  fill-only time @ 100 gpm: {fill_hr:,.1f} hr/week')
print(f'  total scheduled time (fill+nonfill): {total_hr:,.1f} hr/week (slack vs 40h: {FOGG_SCHEDULED_HOURS_PER_WEEK - total_hr:,.1f} hr)')

# Increment summary
inc_year = INCREMENTAL_GAL_PER_YEAR if "INCREMENTAL_GAL_PER_YEAR" in globals() else 4_000_000.0
print(f'Increment: +{inc_year:,.0f} gal/year (Fogg-only)')
print(f'  avg +{inc_year/52:,.0f} gal/week')
print(f'  avg +{(inc_year / FOGG_BATCH_GAL) * total_min_per_batch_median / 60 / 52:,.1f} hr/week of batch-time (using inferred minutes/batch)')

print('Peak-week stress (if incremental peaks with existing seasonality):')
print('  See the "Increment scenario" cell output for worst weeks vs 40h schedule.')

# Bottle constraint check (uses BR receipts supply)
print('Bottle constraint check (022001 RIGS 1-gal bottle):')
peak_supply = float(bottle_weekly['supply_br'].max())
peak_demand_scaled = float(bottle_weekly['demand_scaled'].max())
weeks_over_scaled = bottle_weekly.filter(pl.col('demand_scaled') > pl.col('supply_br')).height
worst_def_scaled = float((bottle_weekly['demand_scaled'] - bottle_weekly['supply_br']).max())

buf = [float(x) for x in bottle_weekly['buffer_scaled'].to_list()]
peak_buf = max(buf) if buf else 0.0
peak_week = int(bottle_weekly['week'][buf.index(peak_buf)]) if buf else None

print(f'  peak weekly demand (scaled): {peak_demand_scaled:,.0f} bottles/week')
print(f'  peak weekly BR supply:       {peak_supply:,.0f} bottles/week')
print(f'  weeks demand > BR supply (scaled): {weeks_over_scaled}')
print(f'  worst week deficit (scaled): {max(worst_def_scaled, 0.0):,.0f} bottles')
if peak_week is not None:
    print(f'  peak buffer needed (scaled): {peak_buf:,.0f} bottles (entering week {peak_week})')

# Storage reminder
print('Storage note:')
print('  The leveling (inventory build) cell estimates peak finished-goods inventory (gal and pallets) if you run at a fixed weekly cap.')


Fogg (WW/AF) demand proxy (SO shipments, dataset year):
  total: 4,144,086 gal
  p95 week: 157,446 gal/week | max week: 160,536 gal/week
Fogg effective throughput (from peak-week overhead inference):
  non-fill minutes per 5,040-gal batch (median): 29.3 min
  effective rate over schedule: 63.2 gpm
  implied 40-hr capacity: 151,791 gal/week
Rule-of-thumb for 100k gal/week on Fogg (using inferred minutes/batch):
  fill-only time @ 100 gpm: 16.7 hr/week
  total scheduled time (fill+nonfill): 26.4 hr/week (slack vs 40h: 13.6 hr)
Increment: +4,000,000 gal/year (Fogg-only)
  avg +76,923 gal/week
  avg +20.3 hr/week of batch-time (using inferred minutes/batch)
Peak-week stress (if incremental peaks with existing seasonality):
  See the "Increment scenario" cell output for worst weeks vs 40h schedule.
Bottle constraint check (022001 RIGS 1-gal bottle):
  peak weekly demand (scaled): 315,490 bottles/week
  peak weekly BR supply:       224,294 bottles/week
  weeks demand > BR supply (scaled): 46

## 7. Export Results

In [24]:
# Export key tables to Excel (multi-sheet)
# Edit OUT if you want the file somewhere else.

from pathlib import Path
import pandas as pd

OUT = DB_DIR / f"line_capacity_results_{YEAR}.xlsx"

# Prefer xlsxwriter if installed; fall back to openpyxl.
try:
    import xlsxwriter  # type: ignore
    _ENGINE = "xlsxwriter"
except Exception:
    _ENGINE = "openpyxl"


def _pd(df: pl.DataFrame) -> pd.DataFrame:
    """Polars -> pandas without requiring pyarrow."""
    return pd.DataFrame(df.to_dicts())


sheets: list[tuple[str, pl.DataFrame]] = [
    ("SO_Line_Summary", line_summary),
    ("SO_Units_Monthly", monthly_pivot),
    ("SO_BottleSize_Mix", size_mix),
    ("Gal_Line_Totals", line_totals),
    ("Gal_Monthly_ByLine", monthly_gal),
    ("Gal_Weekly_ByLine", weekly_gal),
    ("Fogg_Weekly", fogg_weekly),
    ("Fogg_Monthly", fogg_monthly),
    (
        "Fogg_Inc_Uniform",
        u.select(["yr", "week", "gal_plus_inc_uniform", "hours_req", "hours_over_40", "extra_10hr_days"]),
    ),
    (
        "Fogg_Inc_Scaled",
        s.select(["yr", "week", "gal_plus_inc_scaled", "hours_req", "hours_over_40", "extra_10hr_days"]),
    ),
    ("Bottle_BR_Daily", br_daily),
    ("Bottle_BR_Weekly", br_weekly),
    ("Bottle_BR_Monthly", br_monthly),
    ("Bottle_Weekly_Balance", bottle_weekly),
    ("Bottle_Scen_Summary", bottle_scenario_summary),
]

# Excel sheet names max out at 31 chars

def _sheet_name(name: str) -> str:
    return name[:31]


with pd.ExcelWriter(Path(OUT), engine=_ENGINE) as writer:
    for name, df in sheets:
        _pd(df).to_excel(writer, sheet_name=_sheet_name(name), index=False)

print(f"Exported: {OUT}")


Exported: C:\Users\jdavis\Documents\kpk-app\local_machine_scripts\sage100sqlite\db\line_capacity_results_2025.xlsx
